<a href="https://colab.research.google.com/github/Mc-cloud/chessRL/blob/main/agents/Agent_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import sys
sys.path.insert(0, "..")

from utils_NN import (
    UCI_TO_IDX, IDX_TO_UCI, move_to_index, index_to_move,
        board_to_tensor, Node, MCTS, CNN,
        play_one_game, generate_games_with_self_play,
)

In [6]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ChessDataset(Dataset):
    def __init__(self, dataset_total):
        self.dataset = dataset_total

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        state, policy, value = self.dataset[idx]
        return (
            state.clone().detach(),
            torch.tensor(policy, dtype=torch.float32),
            torch.tensor([value], dtype=torch.float32)
        )

def alpha_zero_loss(log_policy_preds, value_preds, policy_targets, value_targets):
  """
  Combine l'erreur sur le score (Value) et l'erreur sur les coups (Policy).
  """
  value_loss = F.mse_loss(value_preds, value_targets)

  policy_loss = -torch.sum(policy_targets * log_policy_preds, dim=1).mean()

  return value_loss + policy_loss

def train_network(neural_net, dataset_total, epochs=10, batch_size=64, learning_rate=0.001):
    """
    Prend le réseau actuel et l'entraîne sur les données du Self-Play.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    neural_net.to(device)

    optimizer = optim.Adam(neural_net.parameters(), lr=learning_rate, weight_decay=1e-4)

    dataset = ChessDataset(dataset_total)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    neural_net.train()

    print(f"Début de l'entraînement sur {device} avec {len(dataset_total)} positions...")

    for epoch in range(epochs):
        total_loss = 0.0

        for states, policies, values in dataloader:
            states = states.to(device)
            policies = policies.to(device)
            values = values.to(device)

            optimizer.zero_grad()

            log_policy_preds, value_preds = neural_net(states)

            loss = alpha_zero_loss(log_policy_preds, value_preds, policies, values)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss moyenne : {total_loss / len(dataloader):.4f}")

    neural_net.to("cpu")
    print("Entraînement terminé !")



In [7]:
import chess
import numpy as np

def evaluate_new_net(old_net, new_net, num_games=10, mcts_simulations=100, win_threshold=0.55, eval_temp_threshold = 6):
    """
    Fait s'affronter l'ancien et le nouveau réseau.
    Retourne True si le nouveau réseau est significativement meilleur.
    """
    print(f"\n⚔️ Bienvenue dans l'Arène ! Début du match en {num_games} parties... ⚔️")
    new_wins = 0
    old_wins = 0
    draws = 0

    old_net.eval()
    new_net.eval()

    for i in range(num_games):
        board = chess.Board()
        move_count = 0

        # on alterne les couleurs.
        if i % 2 == 0:
            white_net = new_net
            black_net = old_net
            new_is_white = True
        else:
            white_net = old_net
            black_net = new_net
            new_is_white = False

        # On instancie des MCTS tout neufs pour vider leur mémoire entre chaque partie
        white_mcts = MCTS(white_net, n_simulations=mcts_simulations)
        black_mcts = MCTS(black_net, n_simulations=mcts_simulations)

        while not board.is_game_over():
            move_count += 1
            if board.turn == chess.WHITE:
                policy_dict = white_mcts.search(board)
            else:
                policy_dict = black_mcts.search(board)

            if move_count <= eval_temp_threshold :
                actions = list(policy_dict.keys())
                probs = list(policy_dict.values())
                best_move = np.random.choice(actions, p = probs)
            else :
                best_move = max(policy_dict, key = policy_dict.get)

            board.push(chess.Move.from_uci(best_move))

        result = board.result()
        if result == "1-0":
            if new_is_white: new_wins += 1
            else: old_wins += 1
        elif result == "0-1":
            if not new_is_white: new_wins += 1
            else: old_wins += 1
        else:
            draws += 1

        print(f"Partie {i+1}/{num_games} terminée | Score global -> Nouveau: {new_wins} | Ancien: {old_wins} | Nuls: {draws}")

    total_score = new_wins + (0.5 * draws)
    win_rate = total_score / num_games

    print(f"\n📊 Ratio de victoire du Challenger : {win_rate:.1%}")

    if win_rate >= win_threshold:
        print("👑 Succès ! Le Nouveau Réseau a surpassé le maître. Il devient le standard.")
        return True
    else:
        print("❌ Échec. Le Nouveau Réseau est rejeté. On garde l'Ancien pour la prochaine génération.")
        return False

In [ ]:
import copy
import os
import torch

os.makedirs("checkpoints", exist_ok=True)

best_network = CNN(input_channels=13, board_size=8, action_size=len(UCI_TO_IDX))

baseline_network = copy.deepcopy(best_network)
torch.save(baseline_network.state_dict(), "checkpoints/gen0_baseline.pt")

history = [] 

iteration = 1
baseline_eval_every = 5 

while True:
    print(f"=== GÉNÉRATION {iteration} ===")
    dataset = generate_games_with_self_play(best_network, num_games=50, num_simulations=100)

    challenger_network = copy.deepcopy(best_network)
    train_network(challenger_network, dataset)

    win_rate_prev = evaluate_new_net(old_net=best_network, new_net=challenger_network)
    print(f"📊 Taux de victoire vs génération précédente : {win_rate_prev:.2%}")

    win_rate_baseline = None
    if iteration % baseline_eval_every == 0:
        win_rate_baseline = evaluate_new_net(old_net=baseline_network, new_net=challenger_network)
        print(f"📈 Taux de victoire vs génération 0 (random) : {win_rate_baseline:.2%}")

    history.append((iteration, win_rate_prev, win_rate_baseline))

    best_network = challenger_network

    checkpoint_path = f"checkpoints/gen{iteration}.pt"
    torch.save(best_network.state_dict(), checkpoint_path)
    print(f"💾 Checkpoint sauvegardé : {checkpoint_path}")

    iteration += 1

=== GÉNÉRATION 1 ===

🔄 Début de la génération de 50 parties en Self-Play (7 workers en parallèle, méthode spawn)...
♟️ Partie 1/50 terminée...
♟️ Partie 2/50 terminée...
♟️ Partie 3/50 terminée...
♟️ Partie 4/50 terminée...
♟️ Partie 5/50 terminée...
♟️ Partie 6/50 terminée...
♟️ Partie 7/50 terminée...
♟️ Partie 8/50 terminée...
♟️ Partie 9/50 terminée...
♟️ Partie 10/50 terminée...
♟️ Partie 11/50 terminée...
♟️ Partie 12/50 terminée...
♟️ Partie 13/50 terminée...
♟️ Partie 14/50 terminée...
♟️ Partie 15/50 terminée...
♟️ Partie 16/50 terminée...
♟️ Partie 17/50 terminée...
♟️ Partie 18/50 terminée...
♟️ Partie 19/50 terminée...
♟️ Partie 20/50 terminée...
♟️ Partie 21/50 terminée...
♟️ Partie 22/50 terminée...
♟️ Partie 23/50 terminée...
♟️ Partie 24/50 terminée...
♟️ Partie 25/50 terminée...
♟️ Partie 26/50 terminée...
♟️ Partie 27/50 terminée...
♟️ Partie 28/50 terminée...
♟️ Partie 29/50 terminée...
♟️ Partie 30/50 terminée...
♟️ Partie 31/50 terminée...
♟️ Partie 32/50 terminée